In [ ]:
:opt 3
:dep kolmox = { path = "../src/lib" }
:dep plotly = "0.13"
:dep tracing-subscriber = "0.3"
:dep .

use kolmox::compress::{
    brotli::CompressBrotli,
    zstd::CompressZstd,
    Compressor,
    cache::{Cache, InMemoryCache}
};
use std::time::{Duration, Instant};

fn read_from_file(file_path: &str) -> String {
    std::fs::read_to_string(file_path).expect("Failed to read file")
}

In [6]:
const FILE_PATH: &str =
    "../../dataset/imdb/list/ls541382956/?ref_=tt_urls_2.html";

println!("Brotli Compression Benchmark");
let page_html: String = read_from_file(FILE_PATH);
let mut results: Vec<benchmarks::BenchmarkResult> = Vec::new();

for quality in 3..11 {
    for lg_window_size in 20..=22 {
        let start = Instant::now();
        let compressor = CompressBrotli::<InMemoryCache>::new(quality, lg_window_size);
        let result = compressor.get_distance(&page_html, &page_html);
        let duration = start.elapsed();

        let benchmark_result = benchmarks::BenchmarkResult {
            quality,
            lg_window_size,
            compression_ratio: result,
            duration,
        };

        results.push(benchmark_result.clone());
        println!("Quality: {quality}, LG Window Size: {lg_window_size}, Compression Ratio: {:.6}, Time: {:?}", result, duration);
    }
}

benchmarks::point_series(&results)

Brotli Compression Benchmark
Quality: 3, LG Window Size: 20, Compression Ratio: 0.003416, Time: 479.666µs
Quality: 3, LG Window Size: 21, Compression Ratio: 0.003416, Time: 97.877µs
Quality: 3, LG Window Size: 22, Compression Ratio: 0.003416, Time: 80.668µs
Quality: 4, LG Window Size: 20, Compression Ratio: 0.000929, Time: 629.392µs
Quality: 4, LG Window Size: 21, Compression Ratio: 0.000929, Time: 145.466µs
Quality: 4, LG Window Size: 22, Compression Ratio: 0.000929, Time: 127.437µs
Quality: 5, LG Window Size: 20, Compression Ratio: 0.000984, Time: 947.063µs
Quality: 5, LG Window Size: 21, Compression Ratio: 0.000984, Time: 178.485µs
Quality: 5, LG Window Size: 22, Compression Ratio: 0.000984, Time: 164.716µs
Quality: 6, LG Window Size: 20, Compression Ratio: 0.000986, Time: 1.39758ms
Quality: 6, LG Window Size: 21, Compression Ratio: 0.000986, Time: 220.834µs
Quality: 6, LG Window Size: 22, Compression Ratio: 0.000986, Time: 198.714µs
Quality: 7, LG Window Size: 20, Compression Ratio

In [7]:
let zstd: CompressZstd<InMemoryCache> = CompressZstd::<InMemoryCache>::recommended();

In [8]:
benchmarks::bench_tests::distance_matrix::heatmap(&zstd, "euronews.com")

In [9]:
benchmarks::bench_tests::distance_matrix::heatmap(&zstd, "amazon")

In [10]:
benchmarks::bench_tests::distance_matrix::heatmap(&zstd, "imdb")

In [11]:
benchmarks::bench_tests::distance_matrix::heatmap(&zstd, "wikipedia")

In [12]:
let wiki_grok: (Vec<String>, Vec<String>, Vec<Vec<f64>>) = benchmarks::bench_tests::wiki_vs_grok::compute_distance_matrix("grokvswiki", &zstd);

In [13]:
benchmarks::bench_tests::wiki_vs_grok::heatmap(&wiki_grok.0, &wiki_grok.1, &wiki_grok.2)

In [14]:
benchmarks::bench_tests::wiki_vs_grok::histogram(&wiki_grok.2)

In [ ]:
let _ = tracing_subscriber::fmt().with_writer(std::io::stdout).try_init();

for ds in ["euronews.com", "amazon", "imdb", "wikipedia"] {
    let compressor = CompressZstd::<InMemoryCache>::recommended();
    benchmarks::bench_tests::bk_tree::bk_tree(compressor, ds, 0.01);
}